In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

In [0]:
# File location and type
file_location = "/FileStore/tables/walmart_stock.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

df.limit(5).display()

Date,Open,High,Low,Close,Volume,Adj Close
2012-01-03,59.970001,61.060001,59.869999,60.330002,12668800,52.619234999999996
2012-01-04,60.209998999999996,60.349998,59.470001,59.709998999999996,9593300,52.078475
2012-01-05,59.349998,59.619999,58.369999,59.419998,12768200,51.825539
2012-01-06,59.419998,59.450001,58.869999,59.0,8069400,51.45922
2012-01-09,59.029999,59.549999,58.919998,59.18,6679300,51.616215000000004


##### What are the column name

In [0]:
df.columns

Out[3]: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']

In [0]:
df.dtypes

Out[4]: [('Date', 'date'),
 ('Open', 'double'),
 ('High', 'double'),
 ('Low', 'double'),
 ('Close', 'double'),
 ('Volume', 'int'),
 ('Adj Close', 'double')]

In [0]:
df.schema.names

Out[5]: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']

##### What dose schema look like

In [0]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Adj Close: double (nullable = true)



##### print out the first 5 columns

In [0]:
df.show(5)

+----------+------------------+---------+---------+------------------+--------+------------------+
|      Date|              Open|     High|      Low|             Close|  Volume|         Adj Close|
+----------+------------------+---------+---------+------------------+--------+------------------+
|2012-01-03|         59.970001|61.060001|59.869999|         60.330002|12668800|52.619234999999996|
|2012-01-04|60.209998999999996|60.349998|59.470001|59.709998999999996| 9593300|         52.078475|
|2012-01-05|         59.349998|59.619999|58.369999|         59.419998|12768200|         51.825539|
|2012-01-06|         59.419998|59.450001|58.869999|              59.0| 8069400|          51.45922|
|2012-01-09|         59.029999|59.549999|58.919998|             59.18| 6679300|51.616215000000004|
+----------+------------------+---------+---------+------------------+--------+------------------+
only showing top 5 rows



In [0]:
for row in df.head(5):
    print(row)
    print('\n')

Row(Date=datetime.date(2012, 1, 3), Open=59.970001, High=61.060001, Low=59.869999, Close=60.330002, Volume=12668800, Adj Close=52.619234999999996)


Row(Date=datetime.date(2012, 1, 4), Open=60.209998999999996, High=60.349998, Low=59.470001, Close=59.709998999999996, Volume=9593300, Adj Close=52.078475)


Row(Date=datetime.date(2012, 1, 5), Open=59.349998, High=59.619999, Low=58.369999, Close=59.419998, Volume=12768200, Adj Close=51.825539)


Row(Date=datetime.date(2012, 1, 6), Open=59.419998, High=59.450001, Low=58.869999, Close=59.0, Volume=8069400, Adj Close=51.45922)


Row(Date=datetime.date(2012, 1, 9), Open=59.029999, High=59.549999, Low=58.919998, Close=59.18, Volume=6679300, Adj Close=51.616215000000004)




In [0]:
df.describe().show()

+-------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|summary|              Open|             High|              Low|            Close|           Volume|        Adj Close|
+-------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|  count|              1258|             1258|             1258|             1258|             1258|             1258|
|   mean| 72.35785375357709|72.83938807631165| 71.9186009594594|72.38844998012726|8222093.481717011|67.23883848728146|
| stddev|  6.76809024470826|6.768186808159218|6.744075756255496|6.756859163732991|  4519780.8431556|6.722609449996857|
|    min|56.389998999999996|        57.060001|        56.299999|        56.419998|          2094900|        50.363689|
|    max|         90.800003|        90.970001|            89.25|        90.470001|         80898100|84.91421600000001|
+-------+------------------+-----------------+--

##### There are too many decimal places for mean and stddev in the describe() dataframe. Format the numbers to just show up to two decimal places. 

In [0]:
df.describe().printSchema()

root
 |-- summary: string (nullable = true)
 |-- Open: string (nullable = true)
 |-- High: string (nullable = true)
 |-- Low: string (nullable = true)
 |-- Close: string (nullable = true)
 |-- Volume: string (nullable = true)
 |-- Adj Close: string (nullable = true)



In [0]:
df_describe=df.describe()
df_describe.select(col('summary'),
                   format_number(col('Open').cast('float'),2).alias('Open'),
                   format_number(col('High').cast('float'),2).alias('High'),
                   format_number(col('Low').cast('float'),2).alias('Low'),
                   format_number(col('Close').cast('float'),2).alias('Close'),
                   col('Volume').cast('int').alias('Volume'),
                   format_number(col('Adj Close').cast('float'),2).alias('Adj Close')
                   ).show()

+-------+--------+--------+--------+--------+--------+---------+
|summary|    Open|    High|     Low|   Close|  Volume|Adj Close|
+-------+--------+--------+--------+--------+--------+---------+
|  count|1,258.00|1,258.00|1,258.00|1,258.00|    1258| 1,258.00|
|   mean|   72.36|   72.84|   71.92|   72.39| 8222093|    67.24|
| stddev|    6.77|    6.77|    6.74|    6.76| 4519780|     6.72|
|    min|   56.39|   57.06|   56.30|   56.42| 2094900|    50.36|
|    max|   90.80|   90.97|   89.25|   90.47|80898100|    84.91|
+-------+--------+--------+--------+--------+--------+---------+



##### Create a new dataframe with a column called HV Ratio that is the ratio of the High Price versus volume of stock traded for a day.

In [0]:
df_HV_Ratio=df.withColumn('HV Ratio',col('High')/col('Volume'))
df_HV_Ratio.select(col('Hv ratio')).show()

+--------------------+
|            Hv ratio|
+--------------------+
|4.819714653321546E-6|
|6.290848613094555E-6|
|4.669412994783916E-6|
|7.367338463826307E-6|
|8.915604778943901E-6|
|8.644477436914568E-6|
|9.351828421515645E-6|
| 8.29141562102703E-6|
|7.712212102001476E-6|
|7.071764823529412E-6|
|1.015495466386981E-5|
|6.576354146362592...|
| 5.90145296180676E-6|
|8.547679455011844E-6|
|8.420709512685392E-6|
|1.041448341728929...|
|8.316075414862431E-6|
|9.721183814992126E-6|
|8.029436027707578E-6|
|6.307432259386365E-6|
+--------------------+
only showing top 20 rows



##### What day had the Peak High in Price?

In [0]:
df.orderBy(col('High').desc()).select(col('Date')).first()
print(row[0]) # Get the first column's value    .first() → Returns a Row object

2012-01-09


##### What is the mean of the Close column?

In [0]:
df.agg(mean(col('Close'))).show()

+-----------------+
|       avg(Close)|
+-----------------+
|72.38844998012726|
+-----------------+



##### What is the max and min of the Volume column?

In [0]:
df.select(max('Volume'), min('Volume')).show()

+-----------+-----------+
|max(Volume)|min(Volume)|
+-----------+-----------+
|   80898100|    2094900|
+-----------+-----------+



In [0]:
max_vol = df.agg(max("Volume")).collect()[0][0]
min_vol = df.agg(min("Volume")).collect()[0][0]

# Filter DataFrame to get the corresponding dates
df.filter((col("Volume") == max_vol) | (col("Volume") == min_vol)).select(['Date','volume']).show()

+----------+--------+
|      Date|  volume|
+----------+--------+
|2013-12-24| 2094900|
|2015-10-14|80898100|
+----------+--------+



##### How many days was the Close lower than 60 dollars?

In [0]:
df.filter('Close <60').count()

Out[17]: 81

In [0]:
df.filter(df['Close'] < 60).count()

Out[18]: 81

##### What percentage of the time was the High greater than 80 dollars ?  In other words, (Number of Days High>80)/(Total Days in the dataset)

In [0]:
df.filter('high>80').count()/df.count()*100

Out[19]: 9.141494435612083

###### What is the Pearson correlation between High and Volume?

In [0]:
df.select(corr('High','volume')).show()

+-------------------+
| corr(High, volume)|
+-------------------+
|-0.3384326061737161|
+-------------------+



##### What is the max High per year?

In [0]:
df_year = df.withColumn("Year",year('date'))

In [0]:
df_year.groupBy('Year').agg(max('High')).show()

+----+---------+
|Year|max(High)|
+----+---------+
|2015|90.970001|
|2013|81.370003|
|2014|88.089996|
|2012|77.599998|
|2016|75.190002|
+----+---------+



In [0]:
window_spec=Window.partitionBy('year').orderBy(col('high').desc())


In [0]:
df_year.withColumn('Rank',row_number().over(window_spec)).\
    filter(col("Rank") == 1).\
    select('year','High').display()

year,High
2012,77.599998
2013,81.370003
2014,88.089996
2015,90.970001
2016,75.190002


##### What is the average Close for each Calendar Month?
In other words, across all the years, what is the average Close price for Jan,Feb, Mar, etc... Your result will have a value for each of these months.


In [0]:
df_month=df.withColumn("Month",month('Date'))

In [0]:
df_month.select('Month','Close').groupBy("Month").agg(avg('Close')).orderBy(col('month')).show(12)

+-----+-----------------+
|Month|       avg(Close)|
+-----+-----------------+
|    1|71.44801958415842|
|    2|  71.306804443299|
|    3|71.77794377570092|
|    4|72.97361900952382|
|    5|72.30971688679247|
|    6| 72.4953774245283|
|    7|74.43971943925233|
|    8|73.02981855454546|
|    9|72.18411785294116|
|   10|71.57854545454543|
|   11| 72.1110893069307|
|   12|72.84792478301885|
+-----+-----------------+



##### What is the average Close for each Month?

In [0]:
df_month_year=df_month.join(df_year,on='date',how='inner')
df_month_year.limit(5).display()

Date,Open,High,Low,Close,Volume,Adj Close,Month,Open,High,Low,Close,Volume,Adj Close,Year
2012-01-03,59.970001,61.060001,59.869999,60.330002,12668800,52.619234999999996,1,59.970001,61.060001,59.869999,60.330002,12668800,52.619234999999996,2012
2012-01-04,60.209998999999996,60.349998,59.470001,59.709998999999996,9593300,52.078475,1,60.209998999999996,60.349998,59.470001,59.709998999999996,9593300,52.078475,2012
2012-01-05,59.349998,59.619999,58.369999,59.419998,12768200,51.825539,1,59.349998,59.619999,58.369999,59.419998,12768200,51.825539,2012
2012-01-06,59.419998,59.450001,58.869999,59.0,8069400,51.45922,1,59.419998,59.450001,58.869999,59.0,8069400,51.45922,2012
2012-01-09,59.029999,59.549999,58.919998,59.18,6679300,51.616215000000004,1,59.029999,59.549999,58.919998,59.18,6679300,51.616215000000004,2012


In [0]:
window_spec2=Window.partitionBy('Year',"month")

In [0]:
df_avg_close=df_month_year.withColumn('Avg_Close', avg(df_month['Close']).over(window_spec2)).\
    select('Month','Year','Avg_Close')


In [0]:
df_avg_close.groupBy('month','year').agg(avg('Avg_Close').alias('Avg Close')).display()

month,year,Avg Close
1,2012,60.2354999
2,2012,60.898
3,2012,60.4336368181818
4,2012,60.149000150000006
5,2012,61.45636340909093
6,2012,67.50380961904762
7,2012,72.40666661904764
8,2012,73.04478265217395
9,2012,74.18157921052628
10,2012,75.3061906190476
